# ❤️ Proyecto: Predicción de Enfermedades Cardiovasculares

## 🎯 Objetivos del Proyecto

En este proyecto práctico aprenderás a:

1. **Trabajar con datos médicos reales** del UCI Heart Disease Dataset
2. **Construir un pipeline completo** de Machine Learning
3. **Evaluar modelos de clasificación binaria** con múltiples métricas
4. **Comparar diferentes algoritmos** (SGD vs Random Forest)
5. **Registrar experimentos** con MLflow de forma profesional
6. **Interpretar resultados médicos** con responsabilidad

---

## ❤️ ¿Por qué es importante?

Las **Enfermedades Cardiovasculares (ECV)** son la principal causa de muerte en el mundo:

- 💔 Causan **17.9 millones** de muertes al año
- ⚠️ **90% son prevenibles** con detección temprana
- 🏥 Un diagnóstico temprano puede **salvar vidas**
- 🤖 La IA puede ayudar a **detectar patrones** que salven vidas

---

## 📊 El Dataset: UCI Heart Disease

- **303 pacientes** con datos clínicos
- **14 características** médicas (edad, presión arterial, colesterol, etc.)
- **Variable objetivo**: Presencia de enfermedad cardíaca (0/1)
- **Tipo de problema**: Clasificación binaria

---

## 🎓 Ejercicio Especial: Completa los Comentarios

**🚨 IMPORTANTE**: A lo largo de este notebook verás comentarios como:

```python
# TODO: [Completa aquí] ¿Qué hace esta función?
```

**Tu misión** es completar estos comentarios explicando:
- ¿Qué hace el código?
- ¿Por qué es importante?
- ¿Qué resultado esperamos?

Esto te ayudará a:
- ✅ Entender profundamente cada paso
- ✅ Practicar documentación de código
- ✅ Prepararte para proyectos reales

---

## 🚀 ¡Empecemos!

## ⚙️ Paso 2: Configuración de MLflow

Configuramos el experimento donde registraremos todos nuestros entrenamientos.

## 📚 Paso 3: Importar Librerías

Importamos todas las herramientas necesarias para el proyecto.

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# TODO: [Completa aquí] ¿Qué debes hacer con esta variable email?
email = 'my.garciaazuara@gmail.com'  # ⚠️ CAMBIAR POR TU EMAIL DE DATABRICKS

# Validación
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
else:
    # TODO: [Completa aquí] ¿Para qué sirve set_tracking_uri?
    # Para especificar a mlflow que use el servidor de tracking de databricks
    mlflow.set_tracking_uri("databricks")
    
    # TODO: [Completa aquí] ¿Qué hace set_experiment?
    # Establece en que experimento guardar las ejecuciones que voy a hacer
    experiment_name = f"/Users/{email}/5-prediccion-infarto"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 70)
    print("✅ MLflow configurado correctamente")
    print("=" * 70)
    print(f"📊 Experimento: {experiment_name}")
    print(f"❤️  Proyecto: Predicción de Enfermedades Cardiovasculares")
    print("=" * 70)

In [0]:
# Librerías básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# TODO: [Completa aquí] ¿Para qué sirve train_test_split?
# Permite dividir los datos de muestra en conjuntos de entrenamiento y de test
from sklearn.model_selection import train_test_split

# TODO: [Completa aquí] ¿Qué es un Pipeline en sklearn?
# Es una cadena de ejecuciones que sigue un orden de ejecucion
from sklearn.pipeline import Pipeline

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# TODO: [Completa aquí] ¿Qué modelos vamos a usar?
# Clasificador del decenso del gradiente estocastico y el bosque aleatorio de varios arboles
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier

# TODO: [Completa aquí] ¿Para qué sirve cross_val_score?
# Para usar validacion cruzada y evaluar mejor el modelo al dividir las mismas muestras en varios folds para entrenar y testear el modelo
from sklearn.model_selection import cross_val_score, cross_val_predict

# TODO: [Completa aquí] ¿Qué métricas usaremos y por qué?
# Precision: cuántos de los casos predichos como positivos son realmente positivos.
# Recall: cuántos de los positivos reales consigue detectar el modelo.
# F1-score: combina precision y recall en una única métrica.
# Matriz de confusión: muestra aciertos y errores por clase.
# ROC-AUC: mide la capacidad del modelo para distinguir entre las clases.
# Accuracy: porcentaje total de predicciones correctas.
# Classification report: resume precision, recall y F1-score para cada clase.
# ROC curve: permite visualizar la relación entre verdaderos positivos y falsos positivos.
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    roc_auc_score,
    accuracy_score,
    classification_report,
    roc_curve
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")

## ❤️ Contexto del Proyecto

### El Problema

En este proyecto trabajaremos con el **UCI Heart Disease Dataset**, uno de los datasets más importantes en medicina predictiva. 

### 📊 Datos Clave sobre ECV

- 💔 **Principal causa de muerte** a nivel mundial
- 📈 **17.9 millones** de muertes anuales
- ⚠️ **90% son prevenibles** con detección temprana
- 🎯 **Diagnóstico temprano** = vidas salvadas
- 🤖 **IA** puede detectar patrones imperceptibles para humanos

### 🎯 Nuestro Objetivo

Construir un modelo de Machine Learning que pueda **predecir la presencia de enfermedad cardiovascular** basándose en características clínicas del paciente.

### ⚕️ Responsabilidad Ética

> ⚠️ **IMPORTANTE**: Este es un proyecto educativo. Los modelos médicos reales requieren:
> - Validación clínica exhaustiva
> - Aprobación regulatoria
> - Supervisión médica profesional
> - Consideraciones éticas y legales

Nuestro objetivo es aprender las técnicas, no reemplazar el criterio médico.

## 📚 Fundamentos: Clasificación Binaria

### 🎯 ¿Qué es Clasificación Binaria?

Un problema donde el modelo debe elegir entre **dos clases**:
- ✅ Clase Positiva (1): Tiene enfermedad cardíaca
- ❌ Clase Negativa (0): No tiene enfermedad cardíaca

---

### 🔄 Pipeline de Entrenamiento

```
1. Preparación de Datos
   ↓
2. Selección del Modelo
   ↓
3. Entrenamiento
   ↓
4. Ajuste de Hiperparámetros
   ↓
5. Evaluación
```

---

### 📊 Métricas de Evaluación

#### 1. **Matriz de Confusión**

|                    | Predicción: No (0) | Predicción: Sí (1) |
|--------------------|--------------------|--------------------|
| **Real: No (0)**   | TN (Verdadero Neg) | FP (Falso Positivo)|
| **Real: Sí (1)**   | FN (Falso Negativo)| TP (Verdadero Pos) |

#### 2. **Precision (Precisión)**
```
Precision = TP / (TP + FP)
```
- "De los que predije como enfermos, ¿cuántos realmente lo están?"
- **Importante cuando**: Los falsos positivos son costosos

#### 3. **Recall (Sensibilidad)**
```
Recall = TP / (TP + FN)
```
- "De todos los enfermos reales, ¿cuántos detecté?"
- **Importante cuando**: No podemos perder ningún caso positivo (medicina)

#### 4. **F1-Score**
```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```
- Balance entre Precision y Recall
- Útil cuando las clases están desbalanceadas

#### 5. **ROC-AUC**
- Curva ROC: Representa el trade-off entre True Positive Rate y False Positive Rate
- AUC: Área bajo la curva (0.5 = aleatorio, 1.0 = perfecto)

#### 6. **Validación Cruzada**
- Divide datos en K partes (folds)
- Entrena K veces, cada vez con un fold diferente como test
- Promedia resultados para obtener estimación robusta

---

### 🎯 ¿Qué métrica usar?

| Situación | Métrica Principal |
|-----------|-------------------|
| Clases balanceadas | Accuracy |
| No perder positivos (medicina) | **Recall** ⭐ |
| Evitar falsos positivos | Precision |
| Balance general | F1-Score |
| Comparar modelos | ROC-AUC |

**En medicina, típicamente priorizamos Recall** porque es mejor detectar un caso falso positivo que perder un verdadero positivo.

## 📁 Paso 4: Cargar y Explorar los Datos

Cargamos el dataset y realizamos un análisis exploratorio inicial.

In [0]:
# TODO: Carga los datos y explica qué contiene el archivo.
data = pd.read_csv("../data/raw/heart.csv")

# TODO: Muestra número de filas, columnas y variable objetivo.
print("Número de filas:", len(data))
print("Número de columnas:", len(data.columns))
print("Variable objetivo: 0 NO hay enfermedad, 1 SI hay enfermedad")
#")

In [0]:
# TODO: Muestra las primeras filas con head().
# TODO: Explica por qué esta inspección es útil antes de modelar.
# Sirve para ver si hemos importado bien los datos y la forma que tienen
data.head()


### 📋 Diccionario de Datos

Cada columna representa una característica médica importante:

#### 👤 Datos Demográficos
1. **age**: Edad del paciente en años
2. **sex**: Sexo (1 = masculino, 0 = femenino)

#### 💊 Síntomas y Diagnóstico
3. **cp**: Tipo de dolor de pecho (Chest Pain)
   - 0: Angina típica
   - 1: Angina atípica
   - 2: Dolor no anginoso
   - 3: Asintomático

4. **exang**: Angina inducida por ejercicio (1 = sí, 0 = no)

#### 🩺 Mediciones Clínicas
5. **trestbps**: Presión arterial en reposo (mm Hg)
6. **chol**: Colesterol sérico (mg/dl)
7. **fbs**: Azúcar en sangre en ayunas > 120 mg/dl (1 = verdadero, 0 = falso)
8. **thalach**: Frecuencia cardíaca máxima alcanzada

#### 📊 Resultados de Pruebas
9. **restecg**: Resultados electrocardiográficos en reposo
   - 0: Normal
   - 1: Anormalidad de onda ST-T
   - 2: Hipertrofia ventricular izquierda

10. **oldpeak**: Depresión del segmento ST inducida por ejercicio

11. **slope**: Pendiente del segmento ST durante ejercicio
    - 0: Ascendente
    - 1: Plano
    - 2: Descendente

12. **ca**: Número de vasos principales coloreados por fluoroscopia (0-3)

13. **thal**: Resultados de prueba de talasemia
    - 1: Normal
    - 2: Defecto fijo
    - 3: Defecto reversible

#### 🎯 Variable Objetivo
14. **target**: Presencia de enfermedad cardíaca
    - 0: No enfermedad
    - 1: Enfermedad presente

---

**💡 Tip**: En medicina, cada una de estas características tiene significado clínico. Un buen data scientist debe entender el dominio del problema.

In [0]:
# TODO: Usa info() para revisar tipos de datos y valores nulos
data.info()

# TODO: Anota qué columnas parecen categóricas y cuáles numéricas.
# Categóricas: sex, cp, fbs, restecg, exang, slope
# Numéricas: age, tresstbps, chol, thalach, oldpeak, ca, thal, target

In [0]:
# TODO: Usa describe() para obtener estadísticas descriptivas.
data.describe()

# TODO: Escribe observaciones sobre rangos, medias y posibles valores atípicos.
# La edad media de los pacientes está en 54 años lo que no es demasiado alta.
# El rango de valores de cada característica es bastante grande
# Las columnas trestbps, chol y thalach tienen escalas de valores mucho mayores que el resto
# Existen valores nulos en las columnas trestbps y chol


### 💡 Observaciones Clave

**TODO: [Completa aquí] Basándote en las estadísticas, ¿qué observaciones puedes hacer sobre:**
- La edad media de los pacientes: La edad meedia de los pacientes está en 54 años lo que no es demasiado alta.
- El rango de valores de cada característica: El rango de valores de cada característica es bastante grande en características como trestbps, chol o thalach.
age: 29-77,
trestbps: 94-200,
chol: 131-564,
thalach: 71-202,
oldpeak: 0-6.2,
ca: 0-4,
thal: 0-3,
target: 0-1
- La presencia de valores nulos: Existen valores nulos en las columnas trestbps y chol
- Características que pueden necesitar normalización: trestbps, chol y thalach

In [0]:
# TODO: Analiza la distribución de la variable objetivo.
target_counts = data.target.value_counts()
print(target_counts)

# TODO: Calcula proporciones por clase.
print(f"Sin enfermedad (0): {target_counts[0]/len(data)*100:.1f}%")
print(f"Con enfermedad (1): {target_counts[1]/len(data)*100:.1f}%")
# TODO: Decide si el dataset está balanceado y explica por qué importa.
# Si, esta balanceado. Es importante porque sino el modelo podria aprender a predecir generalmente la variable mayoritaria ya que en la mayoria de casos acertaria aunque no seria lo correcto.


### 🔍 Análisis Exploratorio Visual

Visualicemos las relaciones entre características para entender mejor los datos.

In [0]:
# TODO: Crea visualizaciones para explorar patrones del dataset.
# Ideas:
# - Distribución de edad por diagnóstico
# - Colesterol vs presión arterial coloreado por target
# - Tipo de dolor de pecho vs diagnóstico
# - Matriz de correlación

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('❤️ Análisis Exploratorio de Datos - Heart Disease', fontsize=16, fontweight='bold')

axes[0, 0].hist([data[data.target==0]['age'], data[data.target==1]['age']], 
                bins=20, label=['Sin enfermedad', 'Con enfermedad'], alpha=0.7)
axes[0, 0].set_xlabel('Edad')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].set_title('Distribución de Edad por Diagnóstico')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

scatter = axes[0, 1].scatter(data['chol'], data['trestbps'], c=data['target'], 
                             alpha=0.6, cmap='RdYlGn_r', edgecolors='black', linewidth=0.5)
axes[0, 1].set_xlabel('Colesterol (mg/dl)')
axes[0, 1].set_ylabel('Presión Arterial en Reposo (mm Hg)')
axes[0, 1].set_title('Colesterol vs Presión Arterial')
plt.colorbar(scatter, ax=axes[0, 1], label='Target')
axes[0, 1].grid(True, alpha=0.3)

cp_target = pd.crosstab(data['cp'], data['target'], normalize='index') * 100
cp_target.plot(kind='bar', ax=axes[1, 0], alpha=0.8)
axes[1, 0].set_xlabel('Tipo de Dolor de Pecho')
axes[1, 0].set_ylabel('Porcentaje (%)')
axes[1, 0].set_title('Tipo de Dolor de Pecho vs Diagnóstico')
axes[1, 0].legend(['Sin enfermedad', 'Con enfermedad'])
axes[1, 0].set_xticklabels(['Típica', 'Atípica', 'No anginoso', 'Asintomático'], rotation=45)
axes[1, 0].grid(True, alpha=0.3, axis='y')

sns.heatmap(data.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', 
            ax=axes[1, 1], cbar_kws={'label': 'Correlación'})
axes[1, 1].set_title('Matriz de Correlación')

plt.tight_layout()
plt.show()

# TODO: Escribe qué patrones observas en los gráficos.
# Entre 40 y 60 años se producen mas infartos
# A mas colesterol y presion arterial, mas infartos
# Los dolores de pecho tipica, atipica y no anginoso hay mas infartos
# Hay baja correlación entre las distintas características

## 🔀 Paso 5: Dividir los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# TODO: Divide los datos en train y test.
# Pistas:
# - Usa train_test_split
# - Define test_size
# - Usa random_state para reproducibilidad

train_set, test_set = train_test_split(data, test_size=0.2, random_state=42, stratify=data.target)

# TODO: Comprueba el tamaño de cada conjunto.
print("Tamaño del conjunto de entrenamiento:", len(train_set))
print("Tamaño del conjunto de test:", len(test_set))

# TODO: Comprueba que las proporciones de target son similares en ambos conjuntos.
print("PROPORCIONES DE TARGET EN EL CONJUNTO DE TRAIN")
print(f"Sin enfermedad (0): {train_set.target.value_counts()[0]/len(train_set)*100:.1f}%")
print(f"Con enfermedad (1): {train_set.target.value_counts()[1]/len(train_set)*100:.1f}%")
print("PROPORCIONES DE TARGET EN EL CONJUNTO DE TEST")
print(f"Sin enfermedad (0): {test_set.target.value_counts()[0]/len(test_set)*100:.1f}%")
print(f"Con enfermedad (1): {test_set.target.value_counts()[1]/len(test_set)*100:.1f}%")


In [0]:
# TODO: Genera histogramas del conjunto de entrenamiento.
train_set.hist(bins=50, figsize=(20, 15))
plt.suptitle('Distribución de Características - Conjunto de Entrenamiento', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

# TODO: Identifica características con escalas o distribuciones muy diferentes.
# age, trestbps, chol y thalach

### 📏 Observación Importante: Escalas Diferentes

**TODO: [Completa aquí] ¿Por qué es un problema que las características tengan escalas diferentes?**
Porque las caracteristicas con mayores valores pueden tener mas influencia que el resto

Revisa columnas como `age`, `trestbps`, `chol`, `thalach` y `oldpeak`.

- **age**: 29-77 
- **trestbps**: 94-200 
- **chol**: 126-564 
- **thalach**: 71-202

**TODO: [Completa aquí] ¿Qué transformación aplicarías para que las variables numéricas sean comparables?**
Standard Scaling para que todas tengan Media = 0 y Desviación estándar = 1.

**TODO: [Completa aquí] ¿Por qué esto ayuda a algunos modelos de Machine Learning?**
Porque normaliza los valores y evita que algunas caracteristicas tengan mas peso que otras unicamente por su escala


## 🔧 Paso 6: Construir el Pipeline de Preprocesamiento

Crearemos un pipeline que prepare los datos automáticamente.

In [0]:
# TODO: Clasifica las columnas en categóricas y numéricas.
cat_attr = ["sex", "cp", "fbs", "restecg", "exang", "slope"]
num_attr = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca", "thal"]

# TODO: Crea un pipeline para variables numéricas.
# Pistas:
# - SimpleImputer para valores nulos
# - StandardScaler para estandarizar
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler())
])

# TODO: Crea un ColumnTransformer que combine numéricas y categóricas.
# Pista: usa OneHotEncoder para variables categóricas.
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attr),      
    ("cat", OneHotEncoder(), cat_attr)    
])

# TODO: Explica por qué usamos One-Hot Encoding para categóricas.
# Porque nos permite convertir las variables categóricas en variables numéricas binarias

## 🎯 Paso 7: Preparar X e y

Separamos características (X) de la variable objetivo (y).

In [0]:
# TODO: Separa características (X) y target (y) del conjunto de entrenamiento.
x_train = train_set.drop("target", axis=1)
y_train = train_set.target

# TODO: Comprueba las dimensiones resultantes.
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

In [0]:
# TODO: Aplica el pipeline al conjunto de entrenamiento.
x_train_pr = full_pipeline.fit_transform(x_train)

# TODO: Explica la diferencia entre fit_transform y transform.
# fit_transform primero aprende de los datos y luego transforma mientras que transform transforma directamente
# TODO: Comprueba cómo cambia el número de columnas.
print("x_train shape:", x_train.shape)
print("x_train_pr shape:", x_train_pr.shape)

## 🤖 Paso 8: Entrenamiento y Evaluación de Modelos

Entrenaremos dos modelos y compararemos sus resultados usando MLflow.

### 🔵 Modelo 1: SGD Classifier (Baseline)

**TODO: [Completa aquí] ¿Qué es SGD (Stochastic Gradient Descent) y cómo funciona?** 
Es un método para entrenar modelos ajustando sus parámetros poco a poco para reducir el error.

Empezaremos con un modelo simple como baseline (línea base) para tener una referencia.

In [0]:
# TODO: Activa autolog de MLflow.
mlflow.sklearn.autolog()

# TODO: Inicia un run para el modelo baseline SGD.
with mlflow.start_run(run_name="SGD Classifier - Baseline") as run:
    # TODO: Crea SGDClassifier.
    sgd_clf = SGDClassifier(random_state=42)

    # TODO: Evalúa con cross_val_score.
    scores = cross_val_score(sgd_clf, x_train_pr, y_train, cv=3, scoring="accuracy")

    # TODO: Registra métricas de validación cruzada en MLflow.
    mlflow.log_metric("cv_accuracy_mean", scores.mean())
    mlflow.log_metric("cv_accuracy_std", scores.std())

    # TODO: Interpreta el resultado obtenido.
    print("Accuracy por fold:")
    for i, score in enumerate(scores, 1):
        print(f"   Fold {i}: {score:.4f} ({score*100:.2f}%)")
    
    print(f"\nAccuracy promedio: {scores.mean():.4f} ± {scores.std():.4f}")
    # Se ha obtenido un accuracy promedio de 80 ± 4% de desviación, lo que indica que el modelo es bueno para predecir si un paciente tiene o no enfermedad coronaria pero no muy bueno.

### 📊 Evaluación del Modelo SGD

In [0]:
# TODO: Obtén predicciones con validación cruzada.
preds = cross_val_predict(sgd_clf, x_train_pr, y_train, cv=3)

# TODO: Explica por qué cross_val_predict es útil para construir métricas y matriz de confusión.
# Porque genera una predicción para cada observación usando un modelo que no ha sido entrenado con esa observación. Así podemos construir la matriz de confusión de manera más realista que prediciendo directamente sobre los mismos datos utilizados para entrenar.

In [0]:
# TODO: Calcula y visualiza la matriz de confusión del modelo SGD.
cm = confusion_matrix(y_train, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Sin enfermedad', 'Con enfermedad'])
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues')
plt.show()

# TODO: Interpreta TN, FP, FN y TP.
print(f"   TN (Verdaderos Negativos): {cm[0,0]} - Sanos correctamente identificados como sanos")
print(f"   FP (Falsos Positivos): {cm[0,1]} - Sanos clasificados como enfermos")
print(f"   FN (Falsos Negativos): {cm[1,0]} - Enfermos clasificados como sanos")
print(f"   TP (Verdaderos Positivos): {cm[1,1]} - Enfermos correctamente identificados como enfermos")
# TODO: ¿Qué tipo de error es más grave en medicina?
# FN ya que indica que un paciente enfermo ha sido clasificado como sano, lo que podría llevar a que no reciba tratamiento.

In [0]:
# TODO: Calcula métricas del modelo SGD.
precision = precision_score(y_train, preds)
recall = recall_score(y_train, preds)
f1 = f1_score(y_train, preds)
roc_auc = roc_auc_score(y_train, preds)

# TODO: Imprime e interpreta las métricas.
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")

# TODO: ¿Qué métrica es más importante en medicina y por qué?
# Recall porque no nos podemos dejar verdaderos enfermos 

### 🌲 Modelo 2: Random Forest Classifier

**TODO: [Completa aquí] ¿Qué es Random Forest y por qué suele funcionar mejor que modelos simples?** Es un modelo formado por muchos arboles de decision que funciona mejor porque combina los resultados de varios arboles Ahora probemos un modelo más potente y comparemos resultados.

Ahora probemos un modelo más potente y comparemos resultados.

In [0]:
# TODO: Entrena y evalúa un Random Forest en un nuevo run de MLflow.
with mlflow.start_run(run_name="Random Forest Classifier") as run:
    rf_clf = RandomForestClassifier(random_state=42)
    rf_scores = cross_val_score(rf_clf, x_train_pr, y_train, cv=3, scoring="accuracy")    
    rf_preds = cross_val_predict(rf_clf, x_train_pr, y_train, cv=3)
    mlflow.log_metric("cv_accuracy_mean", rf_scores.mean())
    mlflow.log_metric("cv_accuracy_std", rf_scores.std())

# TODO: Explica qué hiperparámetros podrías ajustar y por qué comparas contra SGD.
# n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, criterion, class_weight

In [0]:
# TODO: Calcula y visualiza la matriz de confusión de Random Forest.
cm_rf = confusion_matrix(y_train, rf_preds)
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=['Sin enfermedad', 'Con enfermedad'])
fig, ax = plt.subplots(figsize=(8, 6))
disp_rf.plot(ax=ax, cmap='Greens')
plt.title('Matriz de Confusión - Random Forest', fontsize=14, fontweight='bold')
plt.show()

# TODO: Compara los errores con los del modelo SGD.
print(f"TN: {cm_rf[0,0]}")
print(f"FP: {cm_rf[0,1]}")
print(f"FN: {cm_rf[1,0]}")
print(f"TP: {cm_rf[1,1]}")
print(f"\n💡 TODO: [Completa aquí] ¿Mejoró respecto a SGD? ¿En qué?")
# Ha mejorado porque ha detectado 3 TP más y ha bajado 3 FN por lo que ha detectado correctamente el diagnóstico de 3 pacientes más. Sin embargo ha aumentado 1 FP y ha bajado un TN.

In [0]:
# TODO: Calcula métricas de Random Forest.
rf_precision = precision_score(y_train, rf_preds)
rf_recall = recall_score(y_train, rf_preds)
rf_f1 = f1_score(y_train, rf_preds)
rf_roc_auc = roc_auc_score(y_train, rf_preds)

# TODO: Crea una tabla comparativa SGD vs Random Forest.
comparison = pd.DataFrame({
    'Métrica': ['Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'SGD': [precision, recall, f1, roc_auc],
    'Random Forest': [rf_precision, rf_recall, rf_f1, rf_roc_auc],
    'Diferencia': [
        rf_precision - precision,
        rf_recall - recall,
        rf_f1 - f1,
        rf_roc_auc - roc_auc
    ]
})
print(comparison.to_string(index=False))
# TODO: Decide qué modelo funciona mejor y justifica tu respuesta.
# Funciona mejor el del bosque ya que obtiene mejores resultados en todas las métricas menos en precision.

## 🎯 Paso 9: Entrenamiento Final y Evaluación en Test

Ahora entrenaremos el modelo con **todo el conjunto de entrenamiento** y evaluaremos en el conjunto de prueba (que nunca ha visto).

In [0]:
# TODO: Entrena el modelo final usando todo el conjunto de entrenamiento procesado.
forest_clf = RandomForestClassifier(random_state=42)
forest_clf.fit(x_train_pr, y_train)

# TODO: Explica por qué ahora entrenamos con todos los datos de train.
#Porque hasta ahora entrenamos con validacion cruzada

In [0]:
# TODO: Separa características y target del conjunto de prueba.
x_test = test_set.drop("target", axis=1)
y_test = test_set.target

# TODO: Comprueba dimensiones.
print(f"X_test: {x_test.shape}")
print(f"y_test: {y_test.shape}")

In [0]:
# TODO: Transforma el conjunto de prueba y genera predicciones.
x_test_pr = full_pipeline.transform(x_test)
final_preds = forest_clf.predict(x_test_pr)

# TODO: Explica por qué usamos transform() y no fit_transform() en test.
# Porque sino estaríamos utilizando el conjunto de test para entrenar el modelo y queremos que el modelo prediga sobre pacientes que no ha visto.

In [0]:
# TODO: Evalúa resultados finales en el conjunto de prueba.
final_precision = precision_score(y_test, final_preds)
final_recall = recall_score(y_test, final_preds)
final_f1 = f1_score(y_test, final_preds)
final_roc_auc = roc_auc_score(y_test, final_preds)
final_accuracy = accuracy_score(y_test, final_preds)

print(f"Accuracy: {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall: {final_recall:.4f}")
print(f"F1 Score: {final_f1:.4f}")
print(f"ROC-AUC: {final_roc_auc:.4f}")

# TODO: Visualiza la matriz de confusión final.
cm_final = confusion_matrix(y_test, final_preds)
disp_final = ConfusionMatrixDisplay(cm_final, display_labels=['Sin enfermedad', 'Con enfermedad'])
fig, ax = plt.subplots(figsize=(8, 6))
disp_final.plot(ax=ax, cmap='YlGnBu')
plt.title('Matriz de Confusión - Conjunto de Prueba', fontsize=14, fontweight='bold')
plt.show()

# TODO: Genera classification_report.
print(classification_report(y_test, final_preds, 
                          target_names=['Sin enfermedad', 'Con enfermedad'],
                          digits=4))
# TODO: Interpreta si el modelo generaliza bien.
# Si, ya que con los nuevos datos de prueba que el modelo no ha visto, obtiene muy buenos resultados de TP y TN.

## 🎉 Reflexión Final del Proyecto

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [ ] Exploración de datos médicos reales
2. [ ] Pipeline de preprocesamiento
3. [ ] Entrenamiento y comparación de modelos
4. [ ] Evaluación con métricas de clasificación
5. [ ] Validación cruzada
6. [ ] Registro de experimentos en MLflow
7. [ ] Interpretación de resultados con matrices de confusión

### 📊 Conclusiones Clave

**TODO: Basándote en tus resultados, escribe tus conclusiones:**

1. **¿Qué modelo funcionó mejor?**
   - RandomForestClassifier ya que obtuvo mejores métricas salvo la precision.

2. **¿Por qué crees que funcionó mejor?**
   - Porque RandomForest emplea varios árboles de decisión que peuden aprender relaciones no lineales y combina todas las predicciones detectando relaciones entre características que no se ven a simple vista.

3. **¿El modelo es suficientemente bueno para uso médico?**
   - No. Los resultados obtenidos son válidos como ejercicio académico, pero un modelo destinado a uso médico necesitaría validación clínica externa, conjuntos de datos mucho mayores y representativos, análisis de sesgos, validación regulatoria y supervisión de profesionales sanitarios. Además, sería ideal obtener mínimo un 0,95 de detección de verdaderos enfermos ya que ese 0.05 significa mucho.

4. **¿Qué métrica es más importante en este caso?**
   - Recall ya que muestra los verdaderos enfermos que ha detectado
5. **¿Qué mejorarías del modelo?**
   - Buscaría los mejores hiperparámetros a través de una búsqueda en rejilla con GridSearchCV

---

### 💡 Reflexiones Importantes

#### ⚕️ Sobre Medicina e IA

**TODO: Reflexiona sobre:**

1. **Falsos Negativos vs Falsos Positivos**
   - En medicina, ¿cuál es más grave y por qué?
   - Falsos Negativos porque no ha detectado a una persona que está enferma y por tanto no se le aplicará el tratamiento.

2. **Responsabilidad Ética**
   - ¿Puede un modelo de IA tomar decisiones médicas solo?
   - No, ya que no obtiene un 100% de acierots ni es totalmente fiable. Debe tener siempre supervisión de un profesional y utilizarse únicamente como apoyo.

3. **Interpretabilidad**
   - ¿Por qué es importante que los médicos entiendan cómo decide el modelo?
   - Porque sino podrían creer todo lo que devuelve y no tener en cuenta factores que pueden hacer que sea incorrecta la predicción o asunciones que hace el modelo.

---

### 🚀 Desafíos Adicionales

1. **Optimización de Hiperparámetros** con `GridSearchCV`.
2. **Prueba Otros Modelos** como Gradient Boosting, SVM o Logistic Regression.
3. **Feature Engineering** para crear o seleccionar características.
4. **Análisis de Errores** para entender pacientes mal clasificados.
5. **Curva ROC** para comparar thresholds y AUC.


In [0]:
# 🎨 DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del proyecto.

# ========================================
# TODO: DESAFÍO 1 - Grid Search
# ========================================
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='recall',  # Priorizamos recall en medicina
    verbose=1,
    n_jobs=-1
)
grid_search.fit(x_train_pr, y_train)
print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor recall (CV): {grid_search.best_score_:.4f}")
# ========================================
# TODO: DESAFÍO 2 - Curva ROC
# ========================================
from sklearn.metrics import roc_curve, auc
y_proba = forest_clf.predict_proba(x_test_pr)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
# TODO: visualiza la curva ROC.
roc_auc_curve = auc(fpr, tpr)
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc_curve:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.grid(True, alpha=0.3)
plt.show()
# ========================================
# TODO: DESAFÍO 3 - Importancia de características
# ========================================
feature_names = num_attr + list(full_pipeline.named_transformers_['cat'].get_feature_names_out(cat_attr))
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': forest_clf.feature_importances_
}).sort_values('importance', ascending=False)
# TODO: identifica e interpreta las características más importantes.
plt.figure(figsize=(10, 8))
plt.barh(feature_importance.head(10)['feature'], 
         feature_importance.head(10)['importance'])
plt.xlabel('Importancia')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
# ========================================
# TODO: DESAFÍO 4 - Comparar más modelos
# ========================================
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
modelos = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}
resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(x_train_pr, y_train)
    preds = modelo.predict(x_test_pr)
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1_s = f1_score(y_test, preds)
    
    resultados.append({
        'Modelo': nombre,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1_s
    })
df_resultados = pd.DataFrame(resultados).sort_values('Recall', ascending=False)
print(df_resultados.to_string(index=False))